In [21]:
""" Class 16. Multilayer perceptron

Objectives:
1. Understand deep feed forward network
2. How backpropagation works
3. Implement a deep feed forward network using PyTorch
"""
import torch
import torch.nn as nn

In [22]:
""" Deep Neural Network / Artificial Neural Network
1. Feed forward network
   Example: Input layer > Hidden layer (s) > Output layer
2. Convolutional Neural Network
   Example: Input layer (Image) > CNN layer (s) > Fully Connected Layer(s) > output layer
            Image (300 x 300)
            Raw features: 90000 pixels. Each pixel is a feature.
            CNN layers extracts feature maps from the image 
3. Recurrent Neural Network: Sequential data uses forward feed connection
   Hi, His name is Masum. He studies computer science. He ___
"""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 
print(device)

cpu


In [23]:
""" Image -> class

Image: A 2D matrix of pixel values.
Color Channels: 
  - Grayscale image has 1 channel per pixel. For a pixel color between (0-255)
    [ 125 124  15 155
      145 145  245 87]
  - RGB image has 3 channels per pixel. For a pixel (0-255, 0-255, 0-255)
    [ (125, 145, 147) ....
      ...................]        
"""
# Preprocessing pipeline
from torchvision import transforms, datasets

"""
Localized normalization:
   image is normalized with its own mean and std 
Globalized Normalization:
   image is normalized with all the images mean and std
"""

transform = transforms.Compose([
    transforms.ToTensor(), # scales pixels values from 0-255 to 0-1
    transforms.Normalize(mean=(0.1307,), std=(0.3081,))
])

In [24]:
# Download dataset
train_dataset = datasets.MNIST(
    root='E:\\PyCharmProjects\\pythonProject\\test\\data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.MNIST(
    root= 'E:\\PyCharmProjects\\pythonProject\\test\\data',
    train=False,
    download=True,
    transform=transform
)

In [25]:
"""
MNIST: Hand written digit recognition dataset
Trainset contains 60,000 images
Testset contains 10,000 images
Each image is 28x28 in shape
So there is 784 pixels for each image

We are given 785 columns for each image
X: column 1-784 (pixels)
y: column 785 (digits)

Model training steps in each epoch:
   1. Forward propagation / Predicts the outcome given the input values / logits
   2. Calculate the loss
   3. Calculate Gradient w.r.t weights using backpropagation
   4. Update weights 
   
layer 1: w1 
layer 2: y = f(w1)
layer 3: z = f(w2)
loss = loss_fn(z, actual_z)

dJ
___
dw2

dJ      dJ       dw2  
___ = ______ x _____
dw1     dw2      dw1

Batch
=======
We have 60000 training images
We want 10 epochs to run for training

Gradient Descent (GD) is an optimization algorithm that minimizes a model's cost function by updating parameters iteratively. 
Batch GD uses the entire dataset, offering precise convergence but slow speed. 
Stochastic GD (SGD) uses one example per update, allowing fast, frequent updates that are noisy. 
Mini-Batch GD balances these, using small data subsets for faster, stabler convergence, and is the standard approach in deep learning. Batch size 32, 64, 128, 16, 8, 4


for epoch in epochs:
    train_batches = randomly distribute the training samples into batches
    test_batches = randomly distribute the test samples into batches
    for train_batch in train_batches:
        1. Predict the outcome given train_batch inputs 
        2. Calculate loss
        3. Calculate gradients w.r.t weights
        4. Update weights
        5. Validate the performance on the test_batches
        
A batch size is a random sample from the training set.
batch size 32, 64, 128
"""
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
)

In [26]:
class Perceptron(nn.Module):
    def __init__(self, input_size):
        super(Perceptron, self).__init__()
        self.w = nn.Parameter(torch.randn(input_size))
        self.b = nn.Parameter(torch.randn(1))
    
    def forward(self, x):
        x = x @ self.w + self.b
        return x

In [27]:
input_size = 28 * 28
sample_input = torch.randn(input_size)
sample_input

tensor([ 0.1050,  0.7135,  0.2110,  0.8357, -0.1231,  1.2022, -0.7653, -1.4180,
         0.0584,  0.8475, -0.0188,  0.1428, -1.4968,  1.3703, -0.8367,  0.0172,
        -1.0046,  0.7494,  2.4446, -0.1473, -1.5227, -1.9365,  0.3601,  1.3502,
        -1.0284, -0.9440,  0.7701, -0.2924,  1.9092, -2.0187,  2.9481,  1.5267,
        -0.3418, -0.8525,  0.3419,  0.4267,  0.5834, -0.4517, -0.1871,  0.2244,
         1.2758, -0.2893, -0.0065,  0.9965,  0.5473, -0.0775,  2.8710, -0.7914,
         1.0361,  1.7314, -0.3160,  0.6690, -0.7822, -0.3830, -1.4284,  1.6701,
         0.0914,  0.6102, -1.8701, -0.4107,  1.4538, -0.1246, -0.6107,  0.1952,
         1.1372, -0.8520,  0.4643, -1.0241, -0.4537, -1.7024, -0.6706, -1.4238,
         0.1715,  0.8802,  0.2261, -0.3627,  0.8446, -0.3127, -2.2470,  0.2202,
        -1.1817,  0.5434, -0.9447,  0.4485, -0.1969,  0.0830, -1.4844,  2.2856,
         1.1415,  0.0095, -0.2867,  0.2138,  0.6058,  0.1400, -1.0871, -0.7215,
         0.3236,  0.3909, -0.2445,  0.61

In [28]:
perceptron = Perceptron(input_size)
output = perceptron(sample_input)
output

tensor([-25.1290], grad_fn=<AddBackward0>)

In [29]:
class ReLU(nn.Module):
    def __init__(self):
        super(ReLU, self).__init__()
        
    def forward(self, x):
        return torch.maximum(torch.tensor(0.0), x)

In [30]:
relu = ReLU()
output = relu(output)
print(output)

tensor([0.], grad_fn=<MaximumBackward0>)


In [31]:
class Linear(nn.Module):
    def __init__(self, input_size, output_size):
        super(Linear, self).__init__()
        self.perceptrons = nn.ModuleList([
            Perceptron(input_size) for _ in range(output_size)
        ])
    def forward(self, x):
        outputs = [perceptron(x) for perceptron in self.perceptrons]
        outputs = torch.stack(outputs, dim=1)
        return outputs

In [32]:
class DigitClassifier(nn.Module):
    def __init__(self, input_size=784, output_size=10):
        super(DigitClassifier, self).__init__()
        self.fc1 = Linear(input_size, 512)
        self.fc2 = Linear(512, 256)
        self.fc3 = Linear(256, 128)
        self.fc4 = Linear(128, output_size)
        self.relu = nn.ReLU()
    def forward(self, x):
        # print(x.shape) # B, 1, 28, 28 
        x = x.view(-1, input_size) # (B,784)
        # print(x.shape)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.fc4(x)
        return x

In [33]:
""" Use of device

data is stored in (RAM/GPU) when program is running
model is also loaded (RAM/GPU)
"""
model = DigitClassifier(input_size=28 * 28, output_size=10).to(device)
sample_input = sample_input.to(device)

output = model(sample_input)
print(output.shape)
print(output)

torch.Size([1, 10])
tensor([[-24843.1738, -38143.5352, -33574.8750,  -6580.8843,   4583.2939,
          23591.0156,  -1083.7958,  37428.3672,  11379.2783,   7218.8545]],
       grad_fn=<StackBackward0>)


In [34]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

Total trainable parameters: 567,434


In [35]:
(784+1) * 512 + (512+1) * 256 + (256+1) * 128+ (128+1) * 10

567434

In [36]:
param_size = next(model.parameters()).element_size() 
model_size_bytes = total_params * param_size
model_size_mb = model_size_bytes / (1024 * 1024)

In [37]:

print("\n" + "="*50)
print(f"Total trainable parameters : {total_params:,}")
print(f"Parameter size (float32)   : {param_size} bytes")
print(f"Estimated model size       : {model_size_mb:.2f} MB")
print(f"Estimated size (float16)   : {model_size_mb/2:.2f} MB")


Total trainable parameters : 567,434
Parameter size (float32)   : 4 bytes
Estimated model size       : 2.16 MB
Estimated size (float16)   : 1.08 MB


In [38]:
max_val, predicted_id = torch.max(output, 1)
print(max_val)
print("Class label:", predicted_id)

tensor([37428.3672], grad_fn=<MaxBackward0>)
Class label: tensor([7])


In [39]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [40]:
num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device) # ensures model and data are on the same device. i.e. device is of two type cpu, cuda
        outputs = model(data)
        loss = criterion(outputs, target)
        optimizer.zero_grad()
        loss.backward() # compute gradients
        optimizer.step() # update weights
        running_loss += loss.item()
        if batch_idx % 100 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}')
    
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}")

Epoch [1/2], Step [1/938], Loss: 63203.0703
Epoch [1/2], Step [101/938], Loss: 4634.1982
Epoch [1/2], Step [201/938], Loss: 5111.5244


KeyboardInterrupt: 

In [41]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        outputs = model(data)
        _, predicted = torch.max(outputs.data, 1)
        total += target.size(0)
        correct += (predicted == target).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy on the test set: {accuracy:.2f}%')

Accuracy on the test set: 79.72%
